In [3]:
from datetime import datetime, timedelta
from collections import defaultdict

# ---------------------------------------------------------
# 設定・定数定義
# ---------------------------------------------------------
# 1日の引き出し上限額
DAILY_LIMIT = 500000
# 「上限に近い」とみなす閾値（ここでは90%以上の45万円と設定）
SUSPICIOUS_AMOUNT_THRESHOLD = 450000
# 「短い時間」とみなす時間窓（分）
TIME_WINDOW_MINUTES = 30

# ---------------------------------------------------------
# データ前処理・構造化
# 元のリスト atm_withdrawal_records は変更せず、分析用のリストを作成
# ---------------------------------------------------------
# データ構造: {'account': str, 'atm': str, 'dt': datetime, 'amount': int, 'raw_log': list}
processed_records = []

for record in atm_withdrawal_records:
    processed_records.append({
        'account': record[0],
        'atm': record[1],
        'dt': datetime.strptime(record[2], '%Y-%m-%d %H:%M:%S'),
        'amount': record[3],
        'raw_log': record # 出力用に元のリストも保持
    })

# ---------------------------------------------------------
# 【パターン1】の検出ロジック
# 犯人が複数の口座を盗み、一台のATMで短時間に高額引き出しを繰り返す
# ---------------------------------------------------------
pattern1_results = defaultdict(list)

# ATMごとにデータをグループ化
atm_groups = defaultdict(list)
for r in processed_records:
    atm_groups[r['atm']].append(r)

# 各ATM内で時間順にソートし、不審な連鎖を探す
for atm, logs in atm_groups.items():
    logs.sort(key=lambda x: x['dt']) # 時間順に並び替え

    # このATMでの不審なログを一時保存するセット（重複排除のため）
    suspicious_indices = set()

    for i in range(len(logs)):
        # 起点となる取引が高額かチェック
        if logs[i]['amount'] < SUSPICIOUS_AMOUNT_THRESHOLD:
            continue

        # 起点から「短い時間」以内の後続取引をチェック
        for j in range(i + 1, len(logs)):
            time_diff = (logs[j]['dt'] - logs[i]['dt']).total_seconds() / 60

            if time_diff > TIME_WINDOW_MINUTES:
                break # 時間窓を超えたらループ終了

            # 条件合致：後続も高額 かつ 口座番号が異なる（複数口座の悪用）
            if (logs[j]['amount'] >= SUSPICIOUS_AMOUNT_THRESHOLD and
                logs[i]['account'] != logs[j]['account']):

                # パターンに合致するログのインデックスを保存
                suspicious_indices.add(i)
                suspicious_indices.add(j)

    # 検出されたログを結果リストに追加
    if suspicious_indices:
        # インデックス順にログを取得して格納
        sorted_indices = sorted(list(suspicious_indices))
        for idx in sorted_indices:
            pattern1_results[atm].append(logs[idx]['raw_log'])

# ---------------------------------------------------------
# 【パターン2】の検出ロジック
# 一つの口座で、連日上限金額に近い引き出しを行っている
# ---------------------------------------------------------
pattern2_results = defaultdict(list)

# 口座ごとにデータをグループ化
account_groups = defaultdict(list)
for r in processed_records:
    account_groups[r['account']].append(r)

# 各口座の取引を日別に集計
for account, logs in account_groups.items():
    # 日付ごとの合計金額とログを保持する辞書
    daily_summary = defaultdict(lambda: {'total': 0, 'logs': []})

    for log in logs:
        date_str = log['dt'].strftime('%Y-%m-%d')
        daily_summary[date_str]['total'] += log['amount']
        daily_summary[date_str]['logs'].append(log['raw_log'])

    # 日付リストを作成してソート
    sorted_dates = sorted(daily_summary.keys())

    # 連日の高額引き出しをチェック
    suspicious_dates = set()

    for i in range(len(sorted_dates) - 1):
        current_date_str = sorted_dates[i]
        next_date_str = sorted_dates[i+1]

        current_date = datetime.strptime(current_date_str, '%Y-%m-%d')
        next_date = datetime.strptime(next_date_str, '%Y-%m-%d')

        current_amount = daily_summary[current_date_str]['total']
        next_amount = daily_summary[next_date_str]['total']

        # 条件：日付が連続している かつ 両日とも上限に近い額を引き出している
        if ((next_date - current_date).days == 1 and
            current_amount >= SUSPICIOUS_AMOUNT_THRESHOLD and
            next_amount >= SUSPICIOUS_AMOUNT_THRESHOLD):

            suspicious_dates.add(current_date_str)
            suspicious_dates.add(next_date_str)

    # 結果の格納
    if suspicious_dates:
        # 日付順にソートしてログを追加
        for date_str in sorted(list(suspicious_dates)):
            pattern2_results[account].extend(daily_summary[date_str]['logs'])

# ---------------------------------------------------------
# 結果の出力
# ---------------------------------------------------------

print("【パターン1】")
if pattern1_results:
    for atm, logs in pattern1_results.items():
        print(f"ATM: {atm}")
        for log in logs:
            print(log)
else:
    print("該当なし")

print("\n【パターン2】")
if pattern2_results:
    for account, logs in pattern2_results.items():
        print(f"口座番号: {account}")
        for log in logs:
            print(log)
else:
    print("該当なし")

【パターン1】
ATM: ATM_09
['○○○3056', 'ATM_09', '2025-08-20 10:00:00', 485000]
['○○○9624', 'ATM_09', '2025-08-20 10:03:00', 500000]
['○○○1625', 'ATM_09', '2025-08-20 10:06:00', 470000]
['○○○6577', 'ATM_09', '2025-08-20 10:09:00', 490000]
['○○○8172', 'ATM_09', '2025-08-20 10:12:00', 485000]
['○○○2087', 'ATM_09', '2025-08-20 10:15:00', 485000]
['○○○0738', 'ATM_09', '2025-08-20 10:18:00', 500000]
['○○○3060', 'ATM_09', '2025-08-20 10:21:00', 485000]
['○○○2292', 'ATM_09', '2025-08-20 10:24:00', 500000]
['○○○2685', 'ATM_09', '2025-08-20 10:27:00', 470000]

【パターン2】
口座番号: ○○○3791
['○○○3791', 'ATM_01', '2025-08-01 15:46:00', 500000]
['○○○3791', 'ATM_07', '2025-08-02 17:15:00', 500000]
['○○○3791', 'ATM_03', '2025-08-03 17:30:00', 500000]
['○○○3791', 'ATM_06', '2025-08-04 12:36:00', 500000]
['○○○3791', 'ATM_03', '2025-08-05 14:24:00', 500000]
['○○○3791', 'ATM_06', '2025-08-06 15:45:00', 500000]
['○○○3791', 'ATM_07', '2025-08-07 15:54:00', 500000]
['○○○3791', 'ATM_09', '2025-08-08 19:29:00', 500000]
['○

In [2]:
atm_withdrawal_records = [['○○○4941', 'ATM_10', '2025-08-01 09:54:38', 30000], ['○○○4240', 'ATM_03', '2025-08-01 15:03:51', 50000], ['○○○6300', 'ATM_05', '2025-08-01 15:36:07', 170000], ['○○○3791', 'ATM_01', '2025-08-01 15:46:00', 500000], ['○○○1592', 'ATM_10', '2025-08-01 16:19:49', 50000], ['○○○6144', 'ATM_06', '2025-08-01 18:18:04', 7000], ['○○○0722', 'ATM_04', '2025-08-01 23:42:26', 100000], ['○○○9028', 'ATM_02', '2025-08-02 07:23:08', 20000], ['○○○2242', 'ATM_03', '2025-08-02 08:37:41', 110000], ['○○○5721', 'ATM_03', '2025-08-02 16:22:01', 7000], ['○○○3791', 'ATM_07', '2025-08-02 17:15:00', 500000], ['○○○8623', 'ATM_06', '2025-08-03 11:43:50', 30000], ['○○○3791', 'ATM_03', '2025-08-03 17:30:00', 500000], ['○○○3018', 'ATM_01', '2025-08-03 20:22:14', 170000], ['○○○8031', 'ATM_06', '2025-08-03 21:18:22', 170000], ['○○○3550', 'ATM_02', '2025-08-04 02:36:59', 30000], ['○○○6657', 'ATM_08', '2025-08-04 03:55:15', 20000], ['○○○2875', 'ATM_03', '2025-08-04 08:35:14', 50000], ['○○○5097', 'ATM_04', '2025-08-04 11:21:51', 30000], ['○○○3791', 'ATM_06', '2025-08-04 12:36:00', 500000], ['○○○2488', 'ATM_07', '2025-08-04 15:31:40', 10000], ['○○○8776', 'ATM_03', '2025-08-04 17:20:12', 50000], ['○○○7680', 'ATM_06', '2025-08-04 23:39:01', 10000], ['○○○5862', 'ATM_04', '2025-08-05 01:28:03', 50000], ['○○○3974', 'ATM_04', '2025-08-05 04:22:02', 30000], ['○○○8965', 'ATM_01', '2025-08-05 06:01:21', 100000], ['○○○3503', 'ATM_05', '2025-08-05 10:04:51', 20000], ['○○○7842', 'ATM_08', '2025-08-05 12:26:07', 7000], ['○○○3791', 'ATM_03', '2025-08-05 14:24:00', 500000], ['○○○4868', 'ATM_09', '2025-08-06 00:19:12', 100000], ['○○○4664', 'ATM_03', '2025-08-06 02:25:19', 170000], ['○○○7926', 'ATM_04', '2025-08-06 02:41:15', 7000], ['○○○0491', 'ATM_08', '2025-08-06 04:12:38', 10000], ['○○○1898', 'ATM_08', '2025-08-06 11:01:08', 100000], ['○○○3791', 'ATM_06', '2025-08-06 15:45:00', 500000], ['○○○1567', 'ATM_07', '2025-08-06 16:40:33', 10000], ['○○○3588', 'ATM_01', '2025-08-06 20:29:10', 10000], ['○○○1606', 'ATM_02', '2025-08-07 01:58:42', 170000], ['○○○3037', 'ATM_07', '2025-08-07 04:09:26', 50000], ['○○○2900', 'ATM_01', '2025-08-07 09:38:42', 170000], ['○○○7370', 'ATM_10', '2025-08-07 10:06:49', 30000], ['○○○3749', 'ATM_03', '2025-08-07 11:44:30', 50000], ['○○○4263', 'ATM_03', '2025-08-07 12:03:18', 110000], ['○○○8808', 'ATM_05', '2025-08-07 13:16:33', 110000], ['○○○3791', 'ATM_07', '2025-08-07 15:54:00', 500000], ['○○○4824', 'ATM_04', '2025-08-07 17:48:57', 100000], ['○○○5913', 'ATM_04', '2025-08-07 20:12:01', 170000], ['○○○2421', 'ATM_08', '2025-08-07 21:42:04', 170000], ['○○○0862', 'ATM_01', '2025-08-07 21:49:27', 170000], ['○○○9282', 'ATM_04', '2025-08-08 02:36:49', 20000], ['○○○5853', 'ATM_02', '2025-08-08 08:30:38', 30000], ['○○○8441', 'ATM_04', '2025-08-08 12:41:35', 7000], ['○○○9260', 'ATM_06', '2025-08-08 13:33:20', 20000], ['○○○0524', 'ATM_05', '2025-08-08 17:44:30', 30000], ['○○○3791', 'ATM_09', '2025-08-08 19:29:00', 500000], ['○○○9967', 'ATM_09', '2025-08-08 21:38:37', 110000], ['○○○4571', 'ATM_02', '2025-08-09 05:44:20', 20000], ['○○○7232', 'ATM_03', '2025-08-09 12:54:09', 10000], ['○○○3791', 'ATM_07', '2025-08-09 18:28:00', 500000], ['○○○3997', 'ATM_02', '2025-08-10 02:48:49', 10000], ['○○○8702', 'ATM_01', '2025-08-10 08:06:25', 50000], ['○○○6781', 'ATM_10', '2025-08-10 13:09:06', 30000], ['○○○1350', 'ATM_10', '2025-08-10 14:27:45', 50000], ['○○○3791', 'ATM_09', '2025-08-10 15:05:00', 500000], ['○○○8110', 'ATM_09', '2025-08-10 18:45:14', 30000], ['○○○3972', 'ATM_10', '2025-08-10 19:25:33', 110000], ['○○○0675', 'ATM_01', '2025-08-10 20:51:22', 100000], ['○○○0308', 'ATM_09', '2025-08-11 06:22:37', 7000], ['○○○3377', 'ATM_10', '2025-08-11 13:44:26', 50000], ['○○○2032', 'ATM_10', '2025-08-11 19:07:23', 20000], ['○○○3927', 'ATM_07', '2025-08-11 21:36:37', 100000], ['○○○1424', 'ATM_05', '2025-08-12 11:42:32', 20000], ['○○○8026', 'ATM_04', '2025-08-12 13:08:40', 110000], ['○○○4954', 'ATM_08', '2025-08-13 05:21:11', 100000], ['○○○8265', 'ATM_02', '2025-08-13 09:10:01', 50000], ['○○○2498', 'ATM_02', '2025-08-13 11:41:03', 20000], ['○○○3166', 'ATM_10', '2025-08-13 14:15:20', 50000], ['○○○0813', 'ATM_09', '2025-08-13 15:43:35', 10000], ['○○○1634', 'ATM_08', '2025-08-13 17:47:59', 20000], ['○○○8571', 'ATM_02', '2025-08-14 01:49:02', 30000], ['○○○1088', 'ATM_07', '2025-08-14 07:17:26', 20000], ['○○○8958', 'ATM_06', '2025-08-14 10:00:52', 30000], ['○○○3956', 'ATM_04', '2025-08-14 11:59:38', 110000], ['○○○9262', 'ATM_10', '2025-08-14 15:55:28', 100000], ['○○○7908', 'ATM_09', '2025-08-14 20:21:50', 110000], ['○○○6449', 'ATM_06', '2025-08-14 20:47:08', 50000], ['○○○4007', 'ATM_04', '2025-08-15 01:14:23', 20000], ['○○○5445', 'ATM_09', '2025-08-15 03:05:48', 10000], ['○○○8787', 'ATM_06', '2025-08-15 04:01:44', 110000], ['○○○8675', 'ATM_09', '2025-08-15 09:39:24', 30000], ['○○○2700', 'ATM_08', '2025-08-15 10:37:18', 10000], ['○○○7658', 'ATM_05', '2025-08-15 22:39:30', 7000], ['○○○6749', 'ATM_03', '2025-08-15 23:04:33', 7000], ['○○○4228', 'ATM_03', '2025-08-16 01:54:21', 10000], ['○○○0247', 'ATM_07', '2025-08-16 03:33:39', 20000], ['○○○2156', 'ATM_02', '2025-08-16 10:52:22', 7000], ['○○○7731', 'ATM_02', '2025-08-16 19:27:58', 170000], ['○○○7574', 'ATM_01', '2025-08-17 04:50:31', 20000], ['○○○1223', 'ATM_09', '2025-08-17 05:46:32', 30000], ['○○○4494', 'ATM_06', '2025-08-17 11:54:57', 100000], ['○○○0124', 'ATM_10', '2025-08-17 12:25:54', 20000], ['○○○3555', 'ATM_06', '2025-08-17 14:28:29', 170000], ['○○○6102', 'ATM_10', '2025-08-17 15:39:23', 30000], ['○○○4278', 'ATM_08', '2025-08-17 21:44:00', 20000], ['○○○9231', 'ATM_02', '2025-08-17 23:20:28', 7000], ['○○○6036', 'ATM_04', '2025-08-18 16:08:38', 30000], ['○○○2563', 'ATM_03', '2025-08-18 17:07:11', 50000], ['○○○5309', 'ATM_07', '2025-08-18 18:22:24', 170000], ['○○○2081', 'ATM_02', '2025-08-19 01:34:22', 20000], ['○○○1053', 'ATM_01', '2025-08-19 07:03:45', 100000], ['○○○4869', 'ATM_04', '2025-08-19 12:21:58', 7000], ['○○○3305', 'ATM_01', '2025-08-19 13:16:06', 100000], ['○○○6416', 'ATM_03', '2025-08-19 14:17:46', 7000], ['○○○1763', 'ATM_04', '2025-08-19 14:36:19', 20000], ['○○○3248', 'ATM_05', '2025-08-19 15:17:13', 30000], ['○○○9378', 'ATM_02', '2025-08-19 21:02:16', 30000], ['○○○7065', 'ATM_03', '2025-08-20 05:14:54', 30000], ['○○○3307', 'ATM_03', '2025-08-20 05:15:32', 20000], ['○○○3056', 'ATM_09', '2025-08-20 10:00:00', 485000], ['○○○9624', 'ATM_09', '2025-08-20 10:03:00', 500000], ['○○○1625', 'ATM_09', '2025-08-20 10:06:00', 470000], ['○○○6577', 'ATM_09', '2025-08-20 10:09:00', 490000], ['○○○8172', 'ATM_09', '2025-08-20 10:12:00', 485000], ['○○○2087', 'ATM_09', '2025-08-20 10:15:00', 485000], ['○○○0738', 'ATM_09', '2025-08-20 10:18:00', 500000], ['○○○3060', 'ATM_09', '2025-08-20 10:21:00', 485000], ['○○○2292', 'ATM_09', '2025-08-20 10:24:00', 500000], ['○○○2685', 'ATM_09', '2025-08-20 10:27:00', 470000], ['○○○2355', 'ATM_09', '2025-08-20 11:53:34', 7000], ['○○○2639', 'ATM_04', '2025-08-20 23:43:29', 10000], ['○○○2990', 'ATM_07', '2025-08-21 01:36:32', 170000], ['○○○1130', 'ATM_10', '2025-08-21 02:24:21', 50000], ['○○○2335', 'ATM_07', '2025-08-21 06:13:41', 20000], ['○○○2748', 'ATM_04', '2025-08-21 07:22:14', 20000], ['○○○7876', 'ATM_10', '2025-08-21 14:13:28', 100000], ['○○○8631', 'ATM_01', '2025-08-21 19:35:44', 110000], ['○○○5586', 'ATM_07', '2025-08-21 23:22:03', 30000], ['○○○3304', 'ATM_02', '2025-08-22 03:28:17', 50000], ['○○○1302', 'ATM_01', '2025-08-22 06:09:48', 100000], ['○○○7858', 'ATM_04', '2025-08-22 15:03:55', 30000], ['○○○5232', 'ATM_04', '2025-08-22 15:48:24', 50000], ['○○○3454', 'ATM_08', '2025-08-22 18:30:59', 100000], ['○○○8849', 'ATM_08', '2025-08-22 21:17:47', 100000], ['○○○9560', 'ATM_06', '2025-08-22 23:40:18', 110000], ['○○○3089', 'ATM_01', '2025-08-23 03:15:36', 30000], ['○○○1071', 'ATM_04', '2025-08-23 04:10:40', 30000], ['○○○4060', 'ATM_07', '2025-08-23 15:47:07', 170000], ['○○○4815', 'ATM_05', '2025-08-23 21:29:34', 10000], ['○○○9476', 'ATM_06', '2025-08-23 23:59:30', 10000], ['○○○0199', 'ATM_10', '2025-08-24 02:04:39', 20000], ['○○○6695', 'ATM_09', '2025-08-24 03:21:59', 10000], ['○○○9747', 'ATM_06', '2025-08-24 06:43:59', 10000], ['○○○2098', 'ATM_03', '2025-08-24 09:28:35', 100000], ['○○○7620', 'ATM_04', '2025-08-24 11:08:59', 100000], ['○○○9879', 'ATM_06', '2025-08-24 21:18:27', 7000], ['○○○9663', 'ATM_06', '2025-08-25 01:06:39', 7000], ['○○○4514', 'ATM_04', '2025-08-25 03:38:25', 50000], ['○○○0474', 'ATM_01', '2025-08-25 10:00:00', 1000], ['○○○9901', 'ATM_02', '2025-08-25 10:07:00', 1000], ['○○○1996', 'ATM_03', '2025-08-25 10:14:00', 1000], ['○○○2417', 'ATM_04', '2025-08-25 10:21:00', 1000], ['○○○1528', 'ATM_05', '2025-08-25 10:28:00', 1000], ['○○○7972', 'ATM_06', '2025-08-25 10:35:00', 1000], ['○○○0519', 'ATM_07', '2025-08-25 10:42:00', 1000], ['○○○4558', 'ATM_08', '2025-08-25 10:49:00', 1000], ['○○○4036', 'ATM_09', '2025-08-25 10:56:00', 1000], ['○○○6165', 'ATM_10', '2025-08-25 11:03:00', 1000], ['○○○3828', 'ATM_09', '2025-08-25 13:21:44', 7000], ['○○○1850', 'ATM_07', '2025-08-25 14:48:32', 10000], ['○○○9024', 'ATM_09', '2025-08-25 20:38:00', 50000], ['○○○2651', 'ATM_08', '2025-08-25 23:07:59', 30000], ['○○○0306', 'ATM_08', '2025-08-26 03:05:55', 7000], ['○○○5116', 'ATM_01', '2025-08-26 06:44:57', 50000], ['○○○0874', 'ATM_01', '2025-08-26 07:23:53', 10000], ['○○○0236', 'ATM_07', '2025-08-26 15:00:18', 20000], ['○○○2751', 'ATM_04', '2025-08-26 17:50:05', 7000], ['○○○5530', 'ATM_10', '2025-08-26 18:47:01', 20000], ['○○○9565', 'ATM_03', '2025-08-26 20:16:18', 10000], ['○○○1141', 'ATM_08', '2025-08-26 22:23:48', 20000], ['○○○5606', 'ATM_09', '2025-08-26 23:21:35', 10000], ['○○○9552', 'ATM_07', '2025-08-27 05:39:36', 20000], ['○○○4157', 'ATM_06', '2025-08-27 06:18:29', 30000], ['○○○5602', 'ATM_03', '2025-08-27 07:29:42', 10000], ['○○○8063', 'ATM_10', '2025-08-27 12:54:47', 170000], ['○○○2678', 'ATM_10', '2025-08-27 16:39:10', 50000], ['○○○6576', 'ATM_09', '2025-08-27 18:14:19', 10000], ['○○○9900', 'ATM_04', '2025-08-28 00:56:33', 50000], ['○○○3139', 'ATM_08', '2025-08-28 03:54:37', 10000], ['○○○6083', 'ATM_07', '2025-08-28 07:27:56', 20000], ['○○○5283', 'ATM_10', '2025-08-28 07:41:00', 170000], ['○○○4264', 'ATM_03', '2025-08-28 07:42:38', 100000], ['○○○2536', 'ATM_05', '2025-08-28 13:49:48', 30000], ['○○○1270', 'ATM_02', '2025-08-29 00:03:32', 110000], ['○○○7964', 'ATM_08', '2025-08-29 03:00:39', 50000], ['○○○1199', 'ATM_02', '2025-08-29 12:10:31', 50000], ['○○○0899', 'ATM_01', '2025-08-29 17:40:25', 7000], ['○○○3669', 'ATM_06', '2025-08-29 19:13:58', 20000], ['○○○8498', 'ATM_04', '2025-08-29 22:16:16', 50000], ['○○○2739', 'ATM_10', '2025-08-30 02:30:15', 100000], ['○○○0974', 'ATM_03', '2025-08-30 08:23:17', 7000], ['○○○1145', 'ATM_03', '2025-08-30 10:03:06', 170000], ['○○○2404', 'ATM_08', '2025-08-30 11:50:34', 100000], ['○○○8112', 'ATM_03', '2025-08-30 12:41:25', 170000], ['○○○3891', 'ATM_04', '2025-08-30 16:26:00', 30000], ['○○○5027', 'ATM_06', '2025-08-30 19:15:20', 50000], ['○○○4547', 'ATM_02', '2025-08-31 15:56:39', 10000], ['○○○3751', 'ATM_03', '2025-08-31 18:30:22', 7000], ['○○○3638', 'ATM_03', '2025-08-31 19:32:12', 30000], ['○○○0389', 'ATM_01', '2025-08-31 21:50:58', 20000], ['○○○5815', 'ATM_01', '2025-08-31 23:57:20', 20000]]